[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/46_mixed_precision_solution.ipynb)

# 🟡 Solution: Mixed Precision Training Step

Reference solution for a mixed precision training step.

fp16 forward → scaled loss → backward → unscale grads → fp32 update

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def mixed_precision_step(model, optimizer, loss_fn, x, y, loss_scale=1024.0):
    model.half()
    x_fp16 = x.half()
    output = model(x_fp16)
    loss = loss_fn(output.float(), y)
    loss_val = loss.item()

    optimizer.zero_grad()
    (loss * loss_scale).backward()

    for p in model.parameters():
        if p.grad is not None:
            p.grad.data = p.grad.data.float() / loss_scale

    model.float()
    optimizer.step()

    return loss_val

In [ ]:
# Verify: run one step on a small Linear model, print loss and confirm weights updated
import torch.nn as nn

torch.manual_seed(42)
model = nn.Linear(8, 4)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

x = torch.randn(4, 8)
y = torch.randn(4, 4)

weights_before = model.weight.data.clone()

loss_val = mixed_precision_step(model, optimizer, loss_fn, x, y)

print("Loss:", loss_val)
print("Weights updated:", not torch.allclose(model.weight.data, weights_before))
print("Model dtype after step:", model.weight.dtype)

In [ ]:
from torch_judge import check
check("mixed_precision")